In [5]:
wg_sets = [
    {
        "description": "First spatial lag, group concentric rings",
        "variables": {
            "y": "gva",
            "k": "total_assets",
            "l": "employees",
        },
        "type": "g",
        "transforms": {
            "g1": "pc8",
            "g2": {
                "value": "pc4",
                "inner": ["pc8"]
            },
            "g3": {
                "value": "ttwa",
                "inner": ["pc8", "pc4"]
            }
        }
    },
    {
        "description": "All combinations of second-order group spatial lags",
        "variables": [
            "wg1_k", "wg1_l",
            "wg2_k", "wg2_l",
            "wg3_k", "wg3_l"
        ],
        "type": "g",
        "transforms": {
            "g1": "pc8",
            "g2": {
                "value": "pc4",
                "inner": ["pc8"]
            },
            "g3": {
                "value": "ttwa",
                "inner": ["pc8", "pc4"]
            }
        }
    },
    {
        "description": "Third and 4th order group lags for wg1",
        "variables": [
            "w2g1_k", "w2g1_l",
            "w3g1_k", "w3g1_l"
        ],
        "type": "g",
        "transforms": {
            "g1": "pc8"
        }
    },
    {
        "description": "Local and global FE transforms for w2g1",
        "variables": [
            "w2g1_k", "w2g1_l",
            "w3g1_k", "w3g1_l"
        ],
        "type": "g",
        "transforms": {
            "(i-wg1)": {
                "i_minus": True,
                "value": "pc8"
            },
            "(i-vg1)": {
                "i_minus": True,
                "leave_one_out": False,
                "value": "pc8"
            }
        }
    }
]

wd_sets = [
    {
        "description": "Test 1 lag",
        "variables": {
            "y": "gva"
        },
        "type": "n",
        "transforms": [
            "d1"
        ]
    },
    {
        "description": "Distance-weighted spatial lags",
        "variables": {
            "y": "gva",
            "k": "total_assets",
            "l": "employees"
        },
        "type": "n",
        "transforms": [
            "d1",
            "d2",
            "d3"
        ]
    },
    {
        "description": "Higher-index d1 lags",
        "variables": [
            "wd1_k", "wd1_l",
            "w2d1_k", "w2d1_l",
            "w3d1_k", "w3d1_l"
        ],
        "type": "n",
        "transforms": [
            "d1"
        ]
    },
    {
        "description": "Higher-index d2 lags",
        "variables": [
            "wd2_k", "wd2_l"
        ],
        "type": "n",
        "transforms": [
            "d2"
        ]
    },
    {
        "description": "Higher-index d3 lags",
        "variables": [
            "wd3_k", "wd3_l"
        ],
        "type": "n",
        "transforms": [
            "d3"
        ]
    },
    {
        "description": "Local transforms for w2d1",
        "variables": [
            "w2d1_k", "w2d1_l",
            "w3d1_k", "w3d1_l"
        ],
        "type": "n",
        "transforms": {
            "(i-wd1)": {
                "i_minus": True,
                "value": "d1"
            }
        }
    },
    {
        "description": "Global transforms for w2d1",
        "variables": [
            "w2d1_k", "w2d1_l",
            "w3d1_k", "w3d1_l"
        ],
        "type": "n",
        "transforms": {
            "(i-vd1)": {
                "i_minus": True,
                "leave_one_out": False,
                "value": "d1"
            }
        }
    }
]

In [24]:
import re
from site import PREFIXES

# Parses and generates the output column name based on transformation rules.
# Automatically increments indices (e.g., w2g1 -> w3g1) if the transformations match.
def get_out_name(var_name: str, transform_name: str) -> str:
    if "_" in var_name:
        parts = var_name.split("_", 1)
        prefix = parts[0]
        suffix = parts[1]
        
        # Check if the last transformation matches the new one exactly
        match = re.search(r'w(\d+)?' + re.escape(transform_name) + r'$', prefix)

        if transform_name.startswith("(") and transform_name.endswith(")"):
            new_prefix = f"{transform_name}{prefix}"
        elif match:
            num = int(match.group(1)) if match.group(1) else 1
            new_prefix = prefix[:match.start()] + f"w{num + 1}{transform_name}"
        else:
            new_prefix = f"w{transform_name}{prefix}"
            
        return f"{new_prefix}_{suffix}"
    else:
        # Base variable renaming (e.g., 'y' -> 'wg1_y')
        
        if transform_name.startswith("(") and transform_name.endswith(")"):
            return f"{transform_name}{var_name}"
        else:
            return f"w{transform_name}_{var_name}"

transform_tree = { "g": [], "n": [] }
mapping_dict = { "g": {}, "n": {} }
for w_set in wg_sets + wd_sets:
    variables = w_set["variables"]
    transforms = w_set["transforms"]
    w_type = w_set["type"]
    
    # Normalize inputs to dictionaries for uniform loop processing
    if isinstance(transforms, list):
        transforms = {t: t for t in transforms}
    if isinstance(variables, list):
        variables = {v: v for v in variables}

    for new_base, current_col in variables.items():
        for t_name, t_val in transforms.items():
            
            # Parse output string
            out_col = get_out_name(new_base, t_name)
            
            # Parse transformation attributes
            w_col: str                      = t_val
            i_minus: bool                   = False
            leave_one_out: bool             = True
            is_hybrid: bool                 = False
            inner_cols: list[str] | None    = None
            if isinstance(t_val, dict):                
                weight_val = t_val.get("value")
                if type(weight_val) is not str:
                    raise ValueError(f"Expected string for weight value, got {type(weight_val)}: {weight_val}")
                w_col = weight_val
                i_minus = t_val.get("i_minus", False)
                inner_cols = t_val.get("inner", None)
                leave_one_out = t_val.get("leave_one_out", True)
                is_hybrid = "leave_one_out" in t_val  # Triggers overlapping individual graph cluster logic
            
            # Dispatch to appropriate mathematical pipeline 
            if w_type == "g":
                transform_tree["g"].append({
                    "input_col": current_col,
                    "group_col": w_col,
                    "out_col": out_col,
                    "inner_cols": inner_cols,
                    "leave_one_out": leave_one_out,
                    "i_minus": i_minus,
                })
                mapping_dict["g"][out_col] = current_col
            elif w_type == "n":
                transform_tree["n"].append({
                    "input_col": current_col,
                    "weight_col": w_col,
                    "out_col": out_col,
                    "leave_one_out": leave_one_out,
                    "i_minus": i_minus,
                })
                mapping_dict["n"][out_col] = current_col

# Add depth property to each entry in transformation tree
# Based on following the mapping_dict backwards from each out_col to its input_col, counting the number of steps until reaching a base variable (not in mapping_dict).
for w_type in transform_tree:
    for transform in transform_tree[w_type]:
        depth = 1
        current_col = transform["input_col"]
        while current_col in mapping_dict[w_type]:
            current_col = mapping_dict[w_type][current_col]
            depth += 1
        transform["depth"] = depth

print("Transformation tree built successfully.")
# Output transformation tree as json for inspection
import json
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="calculations")
outfile = dirs.tmp_dir / "transform_tree.json"
with open(outfile, "w") as f:
    json.dump(transform_tree, f, indent=4)

# For each depth level, list the variable-transformation pairs that will take place at that depth.
for w_type in transform_tree:
    max_depth = max(t["depth"] for t in transform_tree[w_type]) if transform_tree[w_type] else 0
    print(f"\n{w_type}-type transformations by depth:")
    for depth in range(1, max_depth + 1):
        depth_transforms = [t for t in transform_tree[w_type] if t["depth"] == depth]
        print(f"Depth {depth}: {[f'{t['input_col']} -> {t['out_col']}' for t in depth_transforms]}")

Transformation tree built successfully.

g-type transformations by depth:
Depth 1: ['gva -> wg1_y', 'gva -> wg2_y', 'gva -> wg3_y', 'total_assets -> wg1_k', 'total_assets -> wg2_k', 'total_assets -> wg3_k', 'employees -> wg1_l', 'employees -> wg2_l', 'employees -> wg3_l']
Depth 2: ['wg1_k -> w2g1_k', 'wg1_k -> wg2wg1_k', 'wg1_k -> wg3wg1_k', 'wg1_l -> w2g1_l', 'wg1_l -> wg2wg1_l', 'wg1_l -> wg3wg1_l', 'wg2_k -> wg1wg2_k', 'wg2_k -> w2g2_k', 'wg2_k -> wg3wg2_k', 'wg2_l -> wg1wg2_l', 'wg2_l -> w2g2_l', 'wg2_l -> wg3wg2_l', 'wg3_k -> wg1wg3_k', 'wg3_k -> wg2wg3_k', 'wg3_k -> w2g3_k', 'wg3_l -> wg1wg3_l', 'wg3_l -> wg2wg3_l', 'wg3_l -> w2g3_l']
Depth 3: ['w2g1_k -> w3g1_k', 'w2g1_l -> w3g1_l', 'w2g1_k -> (i-wg1)w2g1_k', 'w2g1_k -> (i-vg1)w2g1_k', 'w2g1_l -> (i-wg1)w2g1_l', 'w2g1_l -> (i-vg1)w2g1_l']
Depth 4: ['w3g1_k -> w4g1_k', 'w3g1_l -> w4g1_l', 'w3g1_k -> (i-wg1)w3g1_k', 'w3g1_k -> (i-vg1)w3g1_k', 'w3g1_l -> (i-wg1)w3g1_l', 'w3g1_l -> (i-vg1)w3g1_l']

n-type transformations by depth:
D

In [ ]:
import ibis
from ibis import _
import pandas as pd
import itertools
from utils.f_0_dirs import get_data_dirs

# Batches multiple group-based spatial lags and processes them in a single Ibis mutation.
# Calculates mutually exclusive donut holes (Inclusion-Exclusion) and (I-W) fixed effects.
def apply_group_W(t_panel: ibis.Table, transforms: list[dict]) -> ibis.Table:

    new_cols = {}
    
    for trans_def in transforms:
        input_col = trans_def["input_col"]
        group_col = trans_def["group_col"]
        out_col = trans_def["out_col"]
        inner_cols = trans_def.get("inner_cols")
        leave_one_out = trans_def.get("leave_one_out", True)
        i_minus = trans_def.get("i_minus", False)

        # 1. Base group sum and count
        sum_col = _[input_col].sum().over(group_by=[_[group_col], _.year])
        count_col = _[input_col].count().over(group_by=[_[group_col], _.year])
        
        # 2. Subtract inner groups if specified (Inclusion-Exclusion Principle)
        if not inner_cols:
            # 3. Standard group (no donut hole)
            if leave_one_out:
                numerator = sum_col - ibis.coalesce(_[input_col], 0)
                denominator = count_col - 1
            else:
                numerator = sum_col
                denominator = count_col
        else:
            sub_sum = None
            sub_count = None
            
            for r in range(1, len(inner_cols) + 1):
                sign = 1 if r % 2 != 0 else -1
                for combo in itertools.combinations(inner_cols, r):
                    part_cols = [_[group_col], _.year] + [_[c] for c in combo]
                    
                    term_sum = _[input_col].sum().over(group_by=part_cols)
                    term_count = _[input_col].count().over(group_by=part_cols)
                    
                    if sub_sum is None:
                        sub_sum = ibis.coalesce(term_sum, 0)
                        sub_count = ibis.coalesce(term_count, 0)
                    else:
                        sub_sum = sub_sum + (ibis.coalesce(term_sum, 0) * sign)
                        sub_count = sub_count + (ibis.coalesce(term_count, 0) * sign)
            
            base_sum = sum_col - sub_sum
            base_count = count_col - sub_count
            
            if leave_one_out:
                numerator = base_sum
                denominator = base_count
            else:
                numerator = base_sum + ibis.coalesce(_[input_col], 0)
                denominator = base_count + 1                
            
        # 4. Row-normalize and apply (I - W) logic
        spatial_lag = ibis.ifelse(denominator > 0, numerator / denominator, ibis.null())
        new_cols[out_col] = _[input_col] - spatial_lag if i_minus else spatial_lag

    # Apply all transformations at this depth in one deferred AST branch
    return t_panel.mutate(**new_cols)


def apply_network_W(t_panel: ibis.Table, t_distance: ibis.Table, transforms: list[dict]) -> ibis.Table:
    """
    Batches multiple distance-decay and overlapping individual network group lags.
    Executes a single structural join per depth level.
    """
    input_cols = list(set(t["input_col"] for t in transforms))

    # 1. Build deferred selections to isolate target and peer data dynamically
    target_selects = {"target_firm": _.registered_number, "target_year": _.year}
    peer_selects = {"peer_firm": _.registered_number, "peer_year": _.year}

    for col in input_cols:
        target_selects[f"target_{col}"] = _[col]
        peer_selects[f"peer_{col}"] = _[col]

    t_target = t_panel.select(**target_selects)
    t_peer = t_panel.select(**peer_selects)

    # 2. Single Master Join (Chain safely evaluates `_` as the growing left-side table)
    t_joined = (
        t_distance
        .inner_join(t_target, _.firm_i == t_target.target_firm)
        .inner_join(t_peer, (_.firm_j == t_peer.peer_firm) & (_.target_year == t_peer.peer_year))
    )

    # 3. Build deferred aggregation dictionary for all transforms concurrently
    agg_exprs = {}
    for trans_def in transforms:
        in_col = trans_def["input_col"]
        w_col = trans_def["weight_col"]
        out_col = trans_def["out_col"]
        leave_one_out = trans_def.get("leave_one_out", True)
        i_minus = trans_def.get("i_minus", False)
        
        is_hybrid = not leave_one_out 

        peer_val = _[f"peer_{in_col}"]
        target_val = _[f"target_{in_col}"]
        weight_val = _[w_col]

        if is_hybrid:
            # Overlapping individual group
            peer_sum = peer_val.sum()
            peer_count = peer_val.count()
            
            numerator = peer_sum + ibis.coalesce(target_val.first(), 0)
            denominator = peer_count + 1
        else:
            # Continuous network distance decay
            weighted_val = peer_val * weight_val
            numerator = weighted_val.sum()
            denominator = ibis.ifelse(peer_val.notnull(), weight_val, ibis.null()).sum()

        spatial_lag = ibis.ifelse(denominator > 0, numerator / denominator, ibis.null())
        agg_exprs[out_col] = target_val.first() - spatial_lag if i_minus else spatial_lag

    # 4. Perform the massive single aggregation
    t_distance_weighted = (
        t_joined
        .group_by([_.firm_i, _.target_year])
        .aggregate(**agg_exprs)
    )

    # 5. Join back and drop redundant identifiers cleanly
    t_panel_mutated = (
        t_panel
        .left_join(
            t_distance_weighted,
            (_.registered_number == t_distance_weighted.firm_i) & (_.year == t_distance_weighted.target_year)
        )
        .drop("firm_i", "target_year")
    )
    
    return t_panel_mutated


# ==========================================
# Schema Iteration & Table Management
# ==========================================

# Assuming 'transform_tree' contains your parsed JSON dict
# transform_tree = json.loads(json_string)

panel_name = "working_yearly"
fixed_name = "working_fixed"
distance_name = "working_distance_ttwa_km"

dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(dirs.db_path)

table_panel = (
    con.table(panel_name)
    .select("registered_number", "year", "gva1", "total_assets", "employees")
    .rename({"gva": "gva1"})
    .distinct(on=["registered_number", "year"])
    .filter(
        (_["gva"].notnull())            & (_["gva"] > 0) &
        (_["total_assets"].notnull())   & (_["total_assets"] > 0) &
        (_["employees"].notnull())      & (_["employees"] > 0)
    )
)

table_fixed = (
    con.table(fixed_name)
    .select("registered_number", "pc8", "pc4", "ttwa")
    .distinct(on="registered_number")
)

# Constrain the universe of distances to only valid firms mapped in the dataset
table_uniques = (
    table_panel
    .inner_join(table_fixed, "registered_number")
    .distinct(on="registered_number")
    .select("registered_number")
)

table_distance = (
    con.table(distance_name)
    .inner_join(table_uniques, _.firm_i == table_uniques.registered_number)
    .mutate(
        d1 = 1 / (_.distance_meters + 1),
        d2 = 1 / (_.distance_meters + 1) ** 2,
        d3 = (-_.distance_meters / 1000).exp()
    )
)

should_run = 'n'

# 1m14s run
if should_run == 'g' or should_run == 'both':
    # Run groups
    t_current_g = table_panel.left_join(table_fixed, "registered_number")
    max_depth_g = max(t.get("depth", 1) for t in transform_tree.get("g", []))
    print(f"\nBeginning group transformations, {len(transform_tree.get('g', []))} required across depth {max_depth_g}.")
    for d in range(1, max_depth_g + 1):
        g_transforms = [t for t in transform_tree.get("g", []) if t.get("depth") == d]
        print(f"--- applying depth {d} with {len(g_transforms)} transformations, {len(t_current_g.columns)} columns...")
        if not g_transforms:
            print(f"No group transformations found at depth {d}.")
            continue
        t_current_g = apply_group_W(t_current_g, g_transforms)
    t_final_g = (
        t_current_g
        .drop("registered_number_right", "pc8", "pc4", "ttwa")
    )
    con.create_table("working_yearly_g", t_final_g, overwrite=True)
    print(con.table("working_yearly_g").sample(0.001).execute())

# 1m24s for depth: 1
# 10 minute run, out of memory
# 1m30s per depth. Should be around 7 minutes total
if should_run == 'n' or should_run == 'both':
    # Run networks
    time_start = pd.Timestamp.now()
    t_start_n = table_panel.left_join(table_fixed, "registered_number")
    max_depth_n = max(t.get("depth", 1) for t in transform_tree.get("n", []))
    print(f"\nBeginning network transformations writing to db at each step, {len(transform_tree.get('n', []))} required across depth {max_depth_n}.")
    write_name = "working_yearly_n"
    
    for d in range(1, max_depth_n + 1):
        try:
            t_imported_n = con.table(write_name)
            t_imported_rows = t_imported_n.count().execute()
            t_imported_columns = t_imported_n.columns
            if t_imported_rows == 0 or len(t_imported_columns) < 5:
                raise ValueError(f"Imported table {write_name} is empty or has insufficient columns.")
            t_current_n = t_imported_n
            time_check = (pd.Timestamp.now() - time_start).total_seconds()
            print(f"[{time_check:.1f}s] --- got table from {write_name} with shape ({t_imported_rows}, {len(t_imported_columns)})")
        except Exception as e:
            t_current_n = t_start_n
            time_check = (pd.Timestamp.now() - time_start).total_seconds()
            print(f"[{time_check:.1f}s] --- starting from t_start_n with shape ({t_start_n.count().execute()}, {len(t_start_n.columns)}) for depth {d} due to: {e}")

        n_transforms = [t for t in transform_tree.get("n", []) if t.get("depth") == d]
        time_check = (pd.Timestamp.now() - time_start).total_seconds()
        print(f"[{time_check:.1f}s] --- applying depth {d} with {len(n_transforms)} transformations, {len(t_current_n.columns)} columns...")
        if not n_transforms:
            print(f"No network transformations found at depth {d}.")
            continue
        t_out_n = apply_network_W(t_current_n, table_distance, n_transforms)
        for col in ["registered_number_right", "pc8", "pc4", "ttwa"]:
            if col in t_out_n.columns:
                t_out_n = t_out_n.drop(col)
        con.create_table("working_yearly_n", t_out_n, overwrite=True)

    print(con.table("working_yearly_n").sample(0.001).execute())



Beginning network transformations writing to db at each step, 28 required across depth 4.
[0.0s] --- starting from t_start_n with shape (1045164, 9) for depth 1 due to: working_yearly_n
[0.2s] --- applying depth 1 with 10 transformations, 9 columns...
[67.5s] --- got table from working_yearly_n with shape (1045164, 18)
[67.5s] --- applying depth 2 with 6 transformations, 18 columns...
[118.1s] --- got table from working_yearly_n with shape (1045164, 24)
[118.1s] --- applying depth 3 with 6 transformations, 24 columns...
[164.8s] --- got table from working_yearly_n with shape (1045164, 30)
[164.8s] --- applying depth 4 with 6 transformations, 30 columns...
     registered_number  year            gva   total_assets  employees  \
0             04659708  2014  158376.715889  201192.442010       2900   
1             SC213584  2007    2572.608701    2799.865706         41   
2             07522037  2021    3325.543350    9706.154716         72   
3             SC116379  2011    1608.867956

In [22]:
# %%script drop table
# Drop table "working_yearly_n" if it exists
import ibis
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(dirs.db_path)

drop = False
if drop:
    if con.table("working_yearly_n").columns:
        con.drop_table("working_yearly_n")
        print("Dropped table 'working_yearly_n' after processing.")

rename = True
if rename:
    table_name = "working_yearly_g"
    in_table = con.table(table_name)
    out_table = (
        in_table
        # .drop('pc8', 'pc4', 'ttwa', 'registered_number_right')
        .drop('registered_number_right')
        .rename({
            # '(i-wd1)w2d1_k': 'w(i-wd1)w2d1_k',
            # '(i-wd1)w2d1_l': 'w(i-wd1)w2d1_l',
            # '(i-vd1)w2d1_k': 'w(i-vd1)w2d1_k',
            # '(i-vd1)w2d1_l': 'w(i-vd1)w2d1_l',
            # '(i-wd1)w3d1_k': 'w(i-wd1)w3d1_k',
            # '(i-wd1)w3d1_l': 'w(i-wd1)w3d1_l',
            # '(i-vd1)w3d1_k': 'w(i-vd1)w3d1_k',
            # '(i-vd1)w3d1_l': 'w(i-vd1)w3d1_l'

            '(i-wg1)w2g1_k': 'w(i-wg1)w2g1_k',
            '(i-wg1)w2g1_l': 'w(i-wg1)w2g1_l',
            '(i-vg1)w2g1_k': 'w(i-vg1)w2g1_k',
            '(i-vg1)w2g1_l': 'w(i-vg1)w2g1_l',
            '(i-wg1)w3g1_k': 'w(i-wg1)w3g1_k',
            '(i-wg1)w3g1_l': 'w(i-wg1)w3g1_l',
            '(i-vg1)w3g1_k': 'w(i-vg1)w3g1_k',
            '(i-vg1)w3g1_l': 'w(i-vg1)w3g1_l'
        })
    )
    print(out_table.columns)
    con.create_table(table_name, out_table, overwrite=True)

('registered_number', 'year', 'gva', 'total_assets', 'employees', 'pc8', 'pc4', 'ttwa', 'wg1_y', 'wg2_y', 'wg3_y', 'wg1_k', 'wg2_k', 'wg3_k', 'wg1_l', 'wg2_l', 'wg3_l', 'w2g1_k', 'wg2wg1_k', 'wg3wg1_k', 'w2g1_l', 'wg2wg1_l', 'wg3wg1_l', 'wg1wg2_k', 'w2g2_k', 'wg3wg2_k', 'wg1wg2_l', 'w2g2_l', 'wg3wg2_l', 'wg1wg3_k', 'wg2wg3_k', 'w2g3_k', 'wg1wg3_l', 'wg2wg3_l', 'w2g3_l', 'w3g1_k', 'w3g1_l', '(i-wg1)w2g1_k', '(i-vg1)w2g1_k', '(i-wg1)w2g1_l', '(i-vg1)w2g1_l', 'w4g1_k', 'w4g1_l', '(i-wg1)w3g1_k', '(i-vg1)w3g1_k', '(i-wg1)w3g1_l', '(i-vg1)w3g1_l')


# 2. Read and sample to verify

In [14]:
t_distance = con.table("working_yearly_n")
print(t_distance.limit(10).execute())
out_file = dirs.tmp_dir / "w_transforms_sample_d.xlsx"
t_distance.sample(0.001).execute().to_excel(out_file, index=False)
print(f"Exported check to {out_file}")

  registered_number  year           gva   total_assets  employees  \
0          14172083  2022  28364.492116  528581.217018        132   
1          12481595  2023   1156.863472   19678.503111         59   
2          02425634  2012   4340.412570    6417.752783        100   
3          02863356  2012    284.375457    7680.100229         16   
4          02796201  2007   2113.558249    2063.751960         34   
5          02994639  2011    101.925862    2073.341440         10   
6          01966923  2006   9551.555727   21174.113669         60   
7          11190178  2022   3494.622750    2735.524518         53   
8          06495303  2012   4086.694177   12733.661312         19   
9          03015047  2016   6389.557116  449015.496222         30   

  registered_number_right       pc8   pc4       ttwa          wd1_y  ...  \
0                14172083  WC1N 3AX  WC1N  E30000234   35218.041366  ...   
1                12481595  WC1N 3AX  WC1N  E30000234   81823.324184  ...   
2           

In [ ]:
t_new = (
    con.table("working_yearly_g")
)
print(f"Total count: {t_new.count().execute():,}, non-null count: {t_new.drop_null().count().execute():,}")
print(t_new.sample(0.001).execute())
t_old = (
    con.table("working_yearly_peers")
    .select(
        "registered_number", "year",
        "gva1_pc8", "gva1_pc4_d", "gva1_ttwa_d",
        "total_assets_pc8", "total_assets_pc4_d", "total_assets_ttwa_d",
        "employees_pc8", "employees_pc4_d", "employees_ttwa_d"
    )
)
t_check = (
    t_new.sample(0.001)
    .select("registered_number", "year", "wg1_y", "wg2_y", "wg3_y", "wg1_k", "wg2_k", "wg3_k", "wg1_l", "wg2_l", "wg3_l")
    .left_join(t_old, ["registered_number", "year"], rname="old_{name}")
    .drop("old_registered_number", "old_year")
    .mutate(
        check_y_pc8 = _.wg1_y == _.gva1_pc8,
        check_y_pc4 = _.wg2_y == _.gva1_pc4_d,
        check_y_ttwa = _.wg3_y == _.gva1_ttwa_d,
        check_k_pc8 = _.wg1_k == _.total_assets_pc8,
        check_k_pc4 = _.wg2_k == _.total_assets_pc4_d,
        check_k_ttwa = _.wg3_k == _.total_assets_ttwa_d,
        check_l_pc8 = _.wg1_l == _.employees_pc8,
        check_l_pc4 = _.wg2_l == _.employees_pc4_d,
        check_l_ttwa = _.wg3_l == _.employees_ttwa_d,
        # round these
        percent_y_pc8 = ((_.wg1_y / _.gva1_pc8 - 1) * 100).round(2),
        percent_y_pc4 = ((_.wg2_y / _.gva1_pc4_d - 1) * 100).round(2),
        percent_y_ttwa = ((_.wg3_y / _.gva1_ttwa_d - 1) * 100).round(2),
        percent_k_pc8 = ((_.wg1_k / _.total_assets_pc8 - 1) * 100).round(2),
        percent_k_pc4 = ((_.wg2_k / _.total_assets_pc4_d - 1) * 100).round(2),
        percent_k_ttwa = ((_.wg3_k / _.total_assets_ttwa_d - 1) * 100).round(2),
        percent_l_pc8 = ((_.wg1_l / _.employees_pc8 - 1) * 100).round(2),
        percent_l_pc4 = ((_.wg2_l / _.employees_pc4_d - 1) * 100).round(2),
        percent_l_ttwa = ((_.wg3_l / _.employees_ttwa_d - 1) * 100).round(2)
    )
)
out_file = dirs.tmp_dir / "w_transforms_sample.xlsx"
t_check.execute().to_excel(out_file, index=False)
print(f"Exported check to {out_file}")

Total count: 1,045,164, non-null count: 694,161
     registered_number  year           gva  total_assets  employees  \
0             SC088529  2012   4266.757504   7949.596244         87   
1             01757853  2019  14927.056603  44405.849348        333   
2             02847178  2019   9167.761395  19722.512456        192   
3             03853667  2011    234.451010   1448.334943         18   
4             03302253  2020    122.701607    978.527614         14   
...                ...   ...           ...           ...        ...   
1041          03742928  2008   9521.248831  10673.899649        106   
1042          04336723  2009    816.239279  15672.926657         34   
1043          04959385  2011   4595.414415  18069.688034         56   
1044          05149751  2019   3850.077118  10558.613801        180   
1045          05325981  2023   1709.107560   5468.257000         77   

     registered_number_right       pc8   pc4       ttwa         wg1_y  ...  \
0                   S

Anomaly on following row:
registered_number	year	wg1_y	wg2_y	wg3_y	wg1_k	wg2_k	wg3_k	wg1_l	wg2_l	wg3_l	gva1_pc8	gva1_pc4_d	gva1_ttwa_d	total_assets_pc8	total_assets_pc4_d	total_assets_ttwa_d	employees_pc8	employees_pc4_d	employees_ttwa_d	check_y_pc8	check_y_pc4	check_y_ttwa	check_k_pc8	check_k_pc4	check_k_ttwa	check_l_pc8	check_l_pc4	check_l_ttwa	percent_y_pc8	percent_y_pc4	percent_y_ttwa	percent_k_pc8	percent_k_pc4	percent_k_ttwa	percent_l_pc8	percent_l_pc4	percent_l_ttwa
04411560	2009	497.241884	622489.0859	60545.99322	11048.83689	1952732.553	635598.9481	28.66666667	3066.189189	693.7905331	677.4403231	622525.7043	80022.57145	359.5797285	1952751.989	722361.3462	13	3066.891892	820.8799609	FALSE	FALSE	FALSE	FALSE	FALSE	FALSE	FALSE	FALSE	FALSE	-26.6	-0.01	-24.34	2972.71	0	-12.01	120.51	-0.02	-15.48